In [ ]:
"""
Grid search using Betti curves on concatenated H0 + H1.

Output:
    - grid_summary_H0H1.csv
"""

import os
import numpy as np
import pandas as pd

from gudhi.representations import BettiCurve
from scipy.stats import mannwhitneyu, combine_pvalues
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import RepeatedStratifiedKFold

# Configuration
BASE = "RIPS"

RESOLUTIONS = [100, 125, 150]
NORMALIZATIONS = ["none", "l1"]
SIGMAS = [0, 1, 2]
THR_OPTIONS = ["0", "p10"]

EPS = 1e-12

N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 0

OUTDIR = "BC_MWU_clean"
os.makedirs(OUTDIR, exist_ok=True)


# DATA LOADING
def read_and_save(filedir, tube):
    """Load Rips diagrams (H0 + H1)."""
    if tube and tube[0] != ".":
        tubenamerips = tube.split("_")[-1].split(".")[0]
        if tubenamerips == "Rips0":

            r0_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips0.txt")
            r1_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips1.txt")

            Rips0 = np.array(pd.read_csv(r0_path, sep=" ", header=None))
            if len(Rips0) and np.isinf(Rips0[-1, 1]):
                Rips0 = Rips0[:-1]

            Rips1 = np.array(pd.read_csv(r1_path, sep=" ", header=None))
            if len(Rips1) and np.isnan(Rips1[-1, 1]):
                Rips1[-1, 1] = 0

            return [[Rips0, Rips1], None, None]

    return []


def list_patients(root_dir):
    return [x for x in sorted(os.listdir(root_dir)) if not x.startswith(".")]


def load_group_diagrams(group_root):
    """Return dict: patient -> [H0, H1]."""
    out = {}
    for patient in list_patients(group_root):
        p_dir = os.path.join(group_root, patient)
        for f in os.listdir(p_dir):
            if f.endswith("_Rips0.txt"):
                data = read_and_save(p_dir, f)
                if data:
                    out[patient] = data[0]
                break
    return out

def collect_persistences(diagrams_list, dim):
    vals = []
    for d in diagrams_list:
        pairs = d[dim]
        if pairs is None or len(pairs) == 0:
            continue
        p = pairs[:, 1] - pairs[:, 0]
        p = p[np.isfinite(p)]
        p = p[p > 0]
        if len(p):
            vals.append(p)
    return np.concatenate(vals) if vals else np.array([])

def apply_threshold(pairs, thr):
    if pairs is None or len(pairs) == 0:
        return np.empty((0, 2))
    pers = pairs[:, 1] - pairs[:, 0]
    return pairs[np.isfinite(pers) & (pers >= thr)]

def betti_curve(pairs, res):
    if pairs is None or len(pairs) == 0:
        return np.zeros(res)
    return BettiCurve(resolution=res).fit_transform([pairs])[0]

def normalize(v, mode):
    if mode == "l1":
        return v / (np.sum(np.abs(v)) + EPS)
    return v

def smooth(v, sigma):
    if sigma > 0:
        return gaussian_filter1d(v, sigma)
    return v

def mannwhitney_per_bin(X0, X1):
    pvals = []
    for j in range(X0.shape[1]):
        _, p = mannwhitneyu(X0[:, j], X1[:, j])
        pvals.append(p)
    return np.array(pvals)

def fisher_combine(pvals):
    pvals = np.clip(pvals, EPS, 1.0)
    return combine_pvalues(pvals)[1]


# Main

def main():

    NR = load_group_diagrams(os.path.join(BASE, "NonRelapse"))
    R  = load_group_diagrams(os.path.join(BASE, "Relapse"))

    X = list(NR.values()) + list(R.values())
    y = np.array([0]*len(NR) + [1]*len(R))

    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE
    )

    summary_rows = []

    for sigma in SIGMAS:
        for thr_opt in THR_OPTIONS:
            for res in RESOLUTIONS:
                for norm in NORMALIZATIONS:

                    scores = []
                    pfishers = []

                    for tr, te in cv.split(np.zeros(len(y)), y):

                        X_tr = [X[i] for i in tr]
                        X_te = [X[i] for i in te]
                        y_te = y[te]

                        # thresholds learned only on TRAIN
                        if thr_opt == "0":
                            thr0 = thr1 = 0.0
                        else:
                            pers0 = collect_persistences(X_tr, 0)
                            pers1 = collect_persistences(X_tr, 1)
                            thr0 = np.percentile(pers0, 10) if len(pers0) else 0.0
                            thr1 = np.percentile(pers1, 10) if len(pers1) else 0.0

                        X_nr, X_r = [], []

                        for d, label in zip(X_te, y_te):

                            # H0
                            c0 = betti_curve(apply_threshold(d[0], thr0), res)
                            c0 = normalize(c0, norm)
                            c0 = smooth(c0, sigma)

                            # H1
                            c1 = betti_curve(apply_threshold(d[1], thr1), res)
                            c1 = normalize(c1, norm)
                            c1 = smooth(c1, sigma)

                            # concatenate both
                            feat = np.concatenate([c0, c1])

                            if label == 0:
                                X_nr.append(feat)
                            else:
                                X_r.append(feat)

                        if len(X_nr) < 2 or len(X_r) < 2:
                            p_fisher = 1.0
                            score = 0.0
                        else:
                            X_nr = np.vstack(X_nr)
                            X_r  = np.vstack(X_r)

                            pvals = mannwhitney_per_bin(X_nr, X_r)
                            p_fisher = fisher_combine(pvals)
                            score = -np.log10(max(p_fisher, EPS))

                        scores.append(score)
                        pfishers.append(p_fisher)

                    summary_rows.append({
                        "dimension": "H0+H1",
                        "sigma": sigma,
                        "thr_opt": thr_opt,
                        "resolution": res,
                        "normalization": norm,
                        "score_median_CV": np.median(scores),
                        "score_mean_CV": np.mean(scores),
                        "p_fisher_median_CV": np.median(pfishers),
                        "p_fisher_mean_CV": np.mean(pfishers),
                        "n_folds": len(scores)
                    })

                    print(f"[H0+H1] sigma={sigma} thr={thr_opt} res={res} norm={norm}")

    df = pd.DataFrame(summary_rows).sort_values(
        ["score_median_CV", "score_mean_CV"],
        ascending=[False, False]
    )

    out = os.path.join(OUTDIR, "grid_summary_H0H1.csv")
    df.to_csv(out, index=False)

    print("Saved:", out)


if __name__ == "__main__":
    main()